# Prompt Engineering 101 - Part V.
## Knowledge & Synthesis (RAG)

---

### *Giving the AI Eyes, Ears, and a Library*

## 1. The Knowledge Cutoff
AI models are frozen in time. They don't know today's news, your private emails, or your company's new policy.
* **Solution:** **RAG (Retrieval Augmented Generation)**.
* **Concept:** Instead of asking the AI to *memorize* facts, we give it an "Open Book" test. We find the relevant page, paste it into the prompt, and say "Answer using this."

## 2. Vector Databases (The "Long-Term Memory")
How do you find the right page in a 1,000-page manual? You can't paste the whole book.
* **Embeddings:** We convert text into numbers (Vectors). Similar concepts have similar numbers.
* **Semantic Search:** Unlike keywords (Ctrl+F), Vector Search understands meaning. It knows that "billing issue" is similar to "invoice error."

## 3. Multi-Modality (The "Eyes")
Modern LLMs can "see." We can send images to the AI by converting them into code (Base64 encoding). This allows us to chat with charts, receipts, and photos.

## 4. Grounding (The "Truth")
Hallucination happens when the AI guesses. **Grounding** forces the AI to cite a specific source (a URL, a file, or a database row). If it can't find the source, it must say "I don't know."


In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini + FAISS + Vision)
# We are installing:
# 1. Google GenAI (The Model)
# 2. FAISS (The Vector Database)
# 3. Sentence-Transformers (The Embedding Tool)

# 1. Install the dependencies
!pip install -q -U google-genai faiss-cpu sentence-transformers


import base64
import requests

import faiss
from sentence_transformers import SentenceTransformer

from google.colab import userdata
from google import genai

from IPython.display import display, Markdown, Image


# 2. Configure the API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# 3. Create a Robust Wrapper Class
class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.5-flash'):
        self.client = genai.Client(api_key=api_key)
        
        # Priority list of models and their availability status
        # True = Available, False = Exhausted (429)
        self.models = {
            'gemini-2.5-flash': True,
            'gemini-2.5-flash-lite': True,
            'gemini-3-flash-preview': True,
        }
        
        # Validation: Ensure the user's choice is valid
        if preferred_model not in self.models:
            raise ValueError(f"Invalid model '{preferred_model}'. "
                             f"Valid options: {list(self.models.keys())}")
            
        self.current_model = preferred_model

    def _get_next_available_model(self):
        """Sets the first model from the non-exhausted models as the currently selected model. 
        Raises error if no available model left."""
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
    
        raise RuntimeError("❌ All available models are currently exhausted. "
                           "Please wait for quotas to reset.")

    def generate_content(self, contents, config=None):
        """
        Recursively attempts to generate content.
        If a model fails with a quota error, it marks it as unavailable 
        and retries with the next available model.
        """
        try:
            # Attempt generation
            response = self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
            return response

        except Exception as e:
            # Check for Rate Limit / Quota errors
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Marking as unavailable...")
                
                # Update State: Mark current as failed
                self.models[self.current_model] = False
                
                # Switch to next available
                self._get_next_available_model()
            
                # Recursive Step: Try again with the new model
                return self.generate_content(contents, config)
            
            # If it's a logic error (e.g. invalid prompt), raise immediately
            raise e

# 4. Initialize
try:
    model = GeminiModel(GEMINI_API_KEY, preferred_model='gemini-2.0-flash')
    print(f"✅ Connection Established. Using {model.current_model}.")
except Exception as e:
    print(f"❌ Error: {e}")

---

### **Phase 1: Manual RAG (Context Stuffing)**

In [ ]:
# @title 📄 Topic 1: The "Open Book" Exam (Manual RAG)
# Concept: The simplest RAG is just pasting the text into the prompt.
# Let's try 3 different domains to see how versatile this is.

# --- 1. LEGAL CONTEXT ---
fake_law = """
THE 'SIESTA ACT' OF 2026
Section 1: All employees are mandated to take a 20-minute nap at 2:00 PM.
Section 2: Coffee is banned after 1:00 PM.
"""

# --- 2. MEDICAL CONTEXT ---
fake_medical = """
PATIENT: John Doe
DIAGNOSIS: Chronic coding fatigue.
PRESCRIPTION: 50mg of Python, taken twice daily.
"""

# --- 3. TECHNICAL CONTEXT ---
fake_tech = """
ERROR CODE 994: "Printer on Fire."
FIX: Unplug immediately. Do not use water. Use Class C extinguisher.
"""

# The Query
prompt = f"""
CONTEXT 1 (Legal): {fake_law}
CONTEXT 2 (Medical): {fake_medical}
CONTEXT 3 (Tech): {fake_tech}

USER QUESTION: "My printer is burning, can I pour water on it?"

INSTRUCTION: Answer based ONLY on the provided context.
"""

print(model.generate_content(prompt).text)

---

### **Phase 2: Vector Databases (The Brain)**

In [ ]:
# @title 🧠 Topic 2: Building the Knowledge Base (FAISS)
# Concept: We can't paste 1000 pages into the prompt. We need a search engine.
# We will build a "Vector Database" that finds the right page for us.

# 1. The Knowledge Base (Imagine this is 1000 items)
documents = [
    "The IT Helpdesk is located on the 3rd floor, Room 302.",
    "Refunds take 5-10 business days to process back to the original card.",
    "The CEO's favorite color is blue.",
    "Fire drills happen every first Monday of the month at 9 AM.",
    "Password resets require a manager's approval via email.",
    "The cafeteria serves sushi on Fridays.",
    "Meeting rooms can be booked via the Outlook plugin."
]

# 2. The Embedder (Converts Text -> Numbers)
print("Loading Embedding Model (Sentence-Transformers)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 3. Create the Database
print("Vectorizing Documents...")
embeddings = embedder.encode(documents)
dimension = embeddings.shape[1]

# Initialize FAISS (The Vector Index)
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"✅ Database Built! Indexed {index.ntotal} documents.")
print(f"Each sentence is now a vector of {dimension} numbers.")

In [ ]:
# @title 🔎 Topic 3: Semantic Search (The Retrieval)
# Concept: Let's search for concepts, NOT keywords.
# Notice: "Money back" finds "Refunds" even though the words don't match.

query = "How do I get my money back?"

# 1. Convert query to numbers
query_vector = embedder.encode([query])

# 2. Search the Index (Find top 2 closest matches)
k = 2
distances, indices = index.search(query_vector, k)

print(f"User Query: '{query}'\n")
print("--- SEARCH RESULTS (Retrieved Context) ---")
for i, idx in enumerate(indices[0]):
    print(f"Match {i+1}: {documents[idx]}")

---

### **Phase 3: The RAG Loop (Putting it Together)**

In [ ]:
# @title 🔄 Topic 4: The Full RAG Loop (Retrieve + Generate)
# Concept: This is the "Magic" step.
# 1. User asks question -> 2. FAISS finds info -> 3. Gemini answers.

user_question = "Where can I get lunch on Friday?"

# STEP 1: RETRIEVE
q_vec = embedder.encode([user_question])
D, I = index.search(q_vec, 1) # Get top 1 result
retrieved_context = documents[I[0][0]]

print(f"1. Retrieved Fact: '{retrieved_context}'")

# STEP 2: GENERATE (Augment the prompt)
rag_prompt = f"""
FACT: {retrieved_context}

QUESTION: {user_question}

ANSWER: (Answer politely based on the fact)
"""

print("\n2. AI Answer:")
print(model.generate_content(rag_prompt).text)

#

In [ ]:
# @title 🚀 LAB 1: The Semantic Helpdesk
# TASK:
# 1. Add a new rule to the database: "The wifi password is 'Guest123!'".
# 2. Ask the AI: "I can't connect to the internet."
# 3. See if the RAG loop works.

# --- STUDENT WORKSPACE ---
new_rule = ["The wifi password is 'Guest123!'"]

# 1. Update Index
new_vec = embedder.encode(new_rule)
index.add(new_vec)
documents.extend(new_rule) # Keep our text list synced!

# 2. The Loop
my_question = "I can't connect to the internet."

# Search
q_v = embedder.encode([my_question])
_, idxs = index.search(q_v, 1)
found_fact = documents[idxs[0][0]]

# Generate
final_prompt = f"Fact: {found_fact}\nQuestion: {my_question}\nAnswer:"
print(model.generate_content(final_prompt).text)
# -------------------------

---

### **Phase 4: Vision (The Eyes)**

In [ ]:
# @title 🖼️ Topic 5: Vision Helper Function
# We need a tool to send images to the AI.
# This function handles the messy parts (Download -> Base64 -> API).

def analyze_image_from_url(image_url, prompt):
    try:
        # 1. Download Image
        response = requests.get(image_url)
        response.raise_for_status()

        # 2. Prepare for API (Gemini expects specific dict format)
        image_part = {
            "mime_type": "image/jpeg",
            "data": base64.b64encode(response.content).decode('utf-8')
        }

        # 3. Send to Model
        print("Analyzing image...")
        model_response = model.generate_content([prompt, image_part])
        return model_response.text

    except Exception as e:
        return f"Error: {e}"

print("✅ Vision Tool Loaded.")

In [ ]:
# @title 🧾 Topic 6: Analyzing Receipts
# Scenario: You have a receipt image. You need the data.

receipt_url = "https://upload.wikimedia.org/wikipedia/commons/0/0b/ReceiptSwiss.jpg"
task = "Extract the Date and Total Amount. Format as JSON."

# Display what the AI sees
display(Image(url=receipt_url, width=200))

# Run Analysis
result = analyze_image_from_url(receipt_url, task)
display(Markdown(result))

In [ ]:
# @title 📊 LAB 2: The Visual Auditor
# TASK: Analyze this chart of Unemployment Rates.
# URL: https://upload.wikimedia.org/wikipedia/commons/8/86/Line_chart_of_unemployment_rates_for_several_nations_1980-2005.png
# Question: "Which country had the highest unemployment in 1995?"

# --- STUDENT WORKSPACE ---
chart_url = "https://upload.wikimedia.org/wikipedia/commons/8/86/Line_chart_of_unemployment_rates_for_several_nations_1980-2005.png"
my_task = "Which country peaks the highest around 1995?"

display(Image(url=chart_url, width=400))
print(analyze_image_from_url(chart_url, my_task))
# -------------------------

---

### **Phase 5: Tools (The Hands)**

In [ ]:
# @title 🌍 Topic 7: Google Search Grounding
# Concept: Giving the AI access to the live internet using the 'Tools' config.

from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# 1. Enable the Search Tool
search_tool = Tool(google_search=GoogleSearch())
search_config = GenerateContentConfig(tools=[search_tool])

# 2. Ask a "Time-Sensitive" Question
# (The AI's training data cuts off in the past, so it needs Search for this)
question = "What is the stock price of Apple (AAPL) right now?"

print(f"Asking: {question}...")
response = model.generate_content(question, config=search_config)
print(response.text)

# Check if it actually used Google
try:
    if response.candidates[0].grounding_metadata.search_entry_point:
        print("\n[✅ Verified: Source found via Google Search]")
except:
    print("\n[⚠️ Warning: No search metadata found]")

In [ ]:
# @title 📈 LAB 3: The Market Researcher
# TASK: Find the expected release date of "GTA VI" (Grand Theft Auto 6).
# Then calculate how many months are left until then.

# --- STUDENT WORKSPACE ---
game_q = "When is GTA VI releasing? How many months from now is that?"
res = model.generate_content(game_q, config=search_config)
print(res.text)
# -------------------------

---

### **Phase 6: NotebookLM (Discussion)**

In [ ]:
# @title 📓 Topic 8: NotebookLM (RAG as a Service)
# Discussion: We just built a RAG system manually (Phase 2 & 3).
# Google's "NotebookLM" is a product that does this automatically for you.

info = """
### What is NotebookLM?
It is a "Virtual Research Assistant" powered by Gemini.

**How it works (The RAG Loop we just built):**
1. **Ingest:** You upload PDFs/Docs (instead of defining a list).
2. **Embed:** It creates a Vector Index (like FAISS) behind the scenes.
3. **Retrieve:** When you chat, it finds relevant pages.
4. **Generate:** It answers using ONLY your sources.

**Key Feature:**
It creates **Audio Overviews** (Podcasts) from your documents.
"""
display(Markdown(info))

---

# 🏠 Homework: The Insurance Claim Bot

### The Scenario
You work at an Insurance Company.
A customer has submitted a claim for a car accident.
They uploaded a photo of the damage.

### The Task
Write a script that:
1.  **Analyzes an Image** of a car crash (use the URL below).
    * *URL:* `https://upload.wikimedia.org/wikipedia/commons/a/a6/Bumper_dent.JPG`
    * *Prompt:* "Identify the specific part of the car that is damaged."
2.  **Searches the Web** for the replacement cost.
    * *Search:* "Replacement cost for [DAMAGED_PART] 2022 Toyota"
3.  **Generates a Report** combining the visual damage and the estimated cost.

In [ ]:
# YOUR CODE GOES HERE